## Amazon Warehouse Dataset Cleanup

This notebook filters the raw Amazon warehouse dataset and produces a clean CSV containing only valid warehouse entries.  
The output file includes:

- <span style="color:blue"><b>Code</b></span> – Warehouse identifier  
- <span style="color:blue"><b>Type</b></span> – Must be **FC** (Fulfillment Center) or **DC** (Distribution Center)  
- <span style="color:blue"><b>Latitude / Longitude</b></span> – Valid geolocation coordinates  

###  Additional Filtering Steps

- Removes rows with missing Type or geolocation values  
- Excludes non-US mainland locations (e.g., Alaska and Hawaii)  

### Final Output File

Code,Type,Latitude,Longitude


In [2]:
import pandas as pd
import requests
from time import sleep
from geopy.geocoders import Nominatim


## Filter out any non 'DC' or 'FC' 

In [ ]:
df = pd.read_csv("../Data/AmazonCenters/amazonCentersUnfilitered.csv")

col = 'Type'

filtered_df = df[df[col].notna() & df[col].str.contains('FC|DC')]

filtered_df.to_csv("../Data/AmazonCenters/amazonCentersFilitered.csv", index=False)


This step loads the original dataset (`amazonCentersUnfilitered.csv`) and filters out only warehouse facilities that operate as:

- **FC – Fulfillment Centers**, which ship individual customer orders  
- **DC – Distribution Centers**, which handle bulk inventory movement

We remove any rows where the `Type` value is empty and keep only entries containing `FC` or `DC`.  
Finally, we export the cleaned dataset as **`amazonCentersFilitered.csv`** for further analysis and mapping.

> This ensures that all remaining locations represent active Amazon logistics centers in the U.S. supply chain.


## Get all Longitude and Latitude of AmazonCentersFiltered.csv

In [ ]:

df = pd.read_csv("../Data/AmazonCenters/amazonCentersFiltered.csv")

geolocator = Nominatim(user_agent="geoapi")

def get_coords(address):
    try:
        location = geolocator.geocode(address)
        if location:
            return pd.Series([location.latitude, location.longitude])
        else:
            return pd.Series([None, None])
    except:
        return pd.Series([None, None])

df[['Latitude', 'Longitude']] = df['Address'].apply(get_coords)

df.to_csv("../Data/AmazonCenters/amazonCenter_with_coords.csv", index=False)


This step converts each facility’s street address into geographic coordinates using the `OpenStreetMap` geocoding service. We define a helper function `get_coords()` that:

- Sends the address to the geocoding API
- Receives a response containing latitude & longitude
- Returns `None` values if no coordinates are found (or an error occurs)

The function is applied to every row in the `Address` column, and two new columns are created:

- `Latitude`
- `Longitude`

Finally, we export the updated dataset as **`amazonCenter_with_coords.csv`**, now with geographic data 

## Output CSV of only Code, Type, Longitude, Latitude

In [ ]:
df = pd.read_csv("../Data/AmazonCenters/amazonCenter_with_coords.csv")

filtered_df = df[
    df['Type'].notna() &
    df['Type'].str.contains('FC|DC', case=False, na=False) &
    df['Latitude'].notna() &
    df['Longitude'].notna() &
    (~df['Location'].isin(['AK', 'HI']))  
]

filtered_df = filtered_df[['Code', 'Type', 'Longitude', 'Latitude']]

filtered_df.to_csv("../AmazonCenters/Data/amazonCenters.csv", index=False)




In this step, we finalize our dataset by applying several filters to remove invalid or unnecessary entries. Specifically, we keep only facilities that:

- Have a valid **Type** labeled as `FC` or `DC`
- Contain valid geographical coordinates (**Latitude & Longitude**)
- Are located within the **continental United States**  
  *(Alaska and Hawaii are excluded to simplify mapping)*

After filtering, we retain only the essential columns for mapping:

- `Code` — facility identifier
- `Type` — FC (Fulfillment Center) or DC (Distribution Center)
- `Longitude`
- `Latitude`

The cleaned dataset is then exported as **`amazonCenters.csv`**, ready for visualization, mapping, and spatial analysis.